# Using ChatOpenAI

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

load_dotenv(override=True)

True

In [7]:
llm = ChatOpenAI(model="gpt-5.4-mini")

message = "In 1 sentence, who won the 2018 F1 Championship"

reply = llm.invoke(message)

print(reply.content)

Lewis Hamilton won the 2018 Formula 1 World Championship.


In [8]:
for chunk in llm.stream("Tell me in two lines how the 2018 Formula 1 World Championship was won"):
    print(chunk.content, end="", flush=True)

Lewis Hamilton won the 2018 Formula 1 World Championship by finishing consistently on the podium and taking 11 race victories.  
He sealed the title with two races to spare at the Mexican Grand Prix, while Ferrari’s Sebastian Vettel and Kimi Räikkönen could no longer catch him.

# Openrouter & the ChatOpenAI object

In [10]:
openrouter_llm = ChatOpenAI(
    model="anthropic/claude-haiku-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

reply = openrouter_llm.invoke("In one sentence, what is the best F1 race win of Lewis Hamilton's career?")
print(reply.content)

Hamilton's 2008 Brazilian Grand Prix victory in the rain at Interlagos, which clinched his first world championship in dramatic fashion, is widely considered his greatest win.


# Messages

In [11]:
messages = [
    SystemMessage("You are an F1 Pundit"),
    HumanMessage("Give a summary in 100 words or less of Lewis Hamilton's 2008 Brazilian Grand Prix victory"),
]

print(llm.invoke(messages).content)

# The exact same call using plain dictionaries, the format you already know
messages_as_dicts = [
    {"role": "system", "content": "You are an F1 Pundit"},
    {"role": "user", "content": "Give a summary in 100 words or less of Lewis Hamilton's 2008 Brazilian Grand Prix victory"},
]
print(llm.invoke(messages_as_dicts).content)

At the 2008 Brazilian Grand Prix, Lewis Hamilton delivered one of Formula 1’s most dramatic title-winning drives. Starting from fourth, he needed to finish fifth or better to secure the championship. On a rain-hit Interlagos circuit, Hamilton passed Toyota’s Timo Glock on the final lap after briefly dropping behind when rain worsened and some cars switched to dry tires. He crossed the line in fifth, just enough to beat Felipe Massa by a single point and become Formula 1 World Champion for the first time, denying Massa victory in his home race and sealing a stunning last-corner title.
Lewis Hamilton’s 2008 Brazilian Grand Prix victory was a dramatic, title-clinching triumph. Going into the final race, he needed at least fifth place to secure the championship, but late rain turned the race chaotic. After briefly dropping back, Hamilton made a crucial pass on Toyota’s Timo Glock in the final corners to retake fifth place, just enough to beat Felipe Massa to the title by one point. While M

# Tools

In [28]:
@tool
def get_f1_standing(driver_surname: str) -> float:
    """Return the current F1 Standings"""
    fake_f1_standings = {"HAMILTON": 1, "NORRIS": 2, "RUSSEL": 3}
    return fake_f1_standings.get(driver_surname.upper(), 'Driver not found')

print("name:", get_f1_standing.name)
print("description:", get_f1_standing.description)
print("args:", get_f1_standing.args)
print("called directly:", get_f1_standing.invoke({"driver_surname": "HAMILTON"}))

name: get_f1_standing
description: Return the current F1 Standings
args: {'driver_surname': {'title': 'Driver Surname', 'type': 'string'}}
called directly: 1


In [29]:
llm_with_tools = llm.bind_tools([get_f1_standing])

response = llm_with_tools.invoke("Where is Hamilton is the F1 Standings? Use the surname in caps when calling the get_f1_standing tool ")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: ''
tool_calls: [{'name': 'get_f1_standing', 'args': {'driver_surname': 'HAMILTON'}, 'id': 'call_a9PcrDF6ajoYTVSABtFbzXoq', 'type': 'tool_call'}]
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 156, 'total_tokens': 180, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EkUwpAXi9plXSVSbl3lig4Zq3Gc4t', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1df7b-40d2-7ea0-b233-df8a907cd3f6-0' tool_calls=[{'name': 'get_f1_standing', 'args': {'driver_surname': 'HAMILTON'}, 'id': 'call_a9PcrDF6ajoYTVSABtFbzXoq', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tok

In [31]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("Where is Hamilton is the F1 Standings? Use the surname in caps when calling the get_f1_standing tool.")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_f1_standing":
        result = get_f1_standing.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

Hamilton is **1st** in the F1 standings.


# Structured Outputs

In [34]:
class Drivers(BaseModel):
    name: str = Field(description="The drivers name")
    position: str = Field(description="Their F1 Standing Position")
    age: int = Field(description="The age of the driver")

structured_llm = llm.with_structured_output(Drivers)

f1_drivers = structured_llm.invoke("Tell me about all the drivers who finished the 2018 season")
print(f1_drivers)
print("Just the name of the drivers:", f1_drivers.name)

name='Lewis Hamilton' position='1st' age=33
Just the name of the drivers: Lewis Hamilton


In [35]:
class Driver(BaseModel):
    name: str = Field(description="The driver's name")
    position: int = Field(description="Final championship position")
    age: int = Field(description="Age at the final race of the 2018 season")

class Drivers(BaseModel):
    drivers: list[Driver] = Field(
        description="All drivers in the final 2018 championship standings"
    )

structured_llm = llm.with_structured_output(Drivers)

f1_drivers = structured_llm.invoke(
    "List all drivers in the final 2018 F1 Drivers' Championship standings. "
    "Give their final position and age at the final race of that season."
)

print(f1_drivers)

print("Just the names:", [driver.name for driver in f1_drivers.drivers])

drivers=[Driver(name='Lewis Hamilton', position=1, age=33), Driver(name='Sebastian Vettel', position=2, age=31), Driver(name='Kimi Räikkönen', position=3, age=38), Driver(name='Max Verstappen', position=4, age=21), Driver(name='Valtteri Bottas', position=5, age=28), Driver(name='Daniel Ricciardo', position=6, age=29), Driver(name='Nico Hülkenberg', position=7, age=31), Driver(name='Sergio Pérez', position=8, age=28), Driver(name='Kevin Magnussen', position=9, age=25), Driver(name='Fernando Alonso', position=10, age=37), Driver(name='Charles Leclerc', position=11, age=21), Driver(name='Carlos Sainz Jr.', position=12, age=24), Driver(name='Romain Grosjean', position=13, age=32), Driver(name='Pierre Gasly', position=14, age=22), Driver(name='Stoffel Vandoorne', position=15, age=26), Driver(name='Esteban Ocon', position=16, age=22), Driver(name='Lance Stroll', position=17, age=20), Driver(name='Marcus Ericsson', position=18, age=28), Driver(name='Brendon Hartley', position=19, age=28), Dri